In [58]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys

pd.set_option("display.max_columns", None)

def find_project_root(marker: str = "requirements.txt") -> Path:
    """Walk upward from the current folder until we find the marker file."""
    current = Path.cwd().resolve()
    for folder in [current, *current.parents]:
        if (folder / marker).exists():
            return folder
    raise FileNotFoundError(f"Could not find '{marker}' in any parent folder.")

PROJECT_ROOT = find_project_root()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("Python:", sys.executable)
print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DATA_DIR)
print("Raw folder exists:", RAW_DATA_DIR.exists())

Python: c:\Users\Windows 10\anaconda3\python.exe
Project root: C:\Users\Windows 10\Desktop\Job based Project\telecom-churn-decision-system
Raw data directory: C:\Users\Windows 10\Desktop\Job based Project\telecom-churn-decision-system\data\raw
Raw folder exists: True


In [59]:
customer_info = pd.read_csv(RAW_DATA_DIR / "customer_info.csv")
usage_data = pd.read_csv(RAW_DATA_DIR / "usage_data.csv")
churn_labels = pd.read_csv(RAW_DATA_DIR / "churn_labels.csv")

print("customer_info:", customer_info.shape)
print("usage_data:", usage_data.shape)
print("churn_labels:", churn_labels.shape)

customer_info: (10200, 7)
usage_data: (120000, 6)
churn_labels: (10000, 2)


In [60]:
# Check the first few rows of each dataset

print("CUSTOMER INFO")
display(customer_info.head())

print("\nUSAGE DATA")
display(usage_data.head())

print("\nCHURN LABELS")
display(churn_labels.head())

CUSTOMER INFO


,CustomerID,Age,Gender,Region,ContractType,MonthlyCharges,SignupDate
0,CUST_0,NaN,Male,Urban,Yearly,48.18,2021-03-20
1,CUST_1,NaN,Female,Suburban,Prepaid,38.94,2020-01-25
2,CUST_2,46.0,Female,Rural,Monthly,48.56,2022-08-31
3,CUST_3,NaN,Female,Urban,Prepaid,24.22,2018-12-09
4,CUST_4,60.0,Male,Rural,Monthly,67.86,2022-12-28



USAGE DATA


,CustomerID,Month,CallMinutes,DataUsageGB,SMSCount,Complaints
0,CUST_0,2023-01-31,518.12,7.21,9,1
1,CUST_0,2023-02-28,439.79,5.99,35,1
2,CUST_0,2023-03-31,545.06,14.16,14,1
3,CUST_0,2023-04-30,461.78,6.66,40,0
4,CUST_0,2023-05-31,434.82,12.75,26,2



CHURN LABELS


,CustomerID,Churn
0,CUST_0,0
1,CUST_1,0
2,CUST_2,0
3,CUST_3,0
4,CUST_4,0


In [61]:
# Check column names and data types

print("CUSTOMER INFO")
customer_info.info()

print("\nUSAGE DATA")
usage_data.info()

print("\nCHURN LABELS")
churn_labels.info()

CUSTOMER INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10200 entries, 0 to 10199
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   CustomerID      10200 non-null  object 
 1   Age             6617 non-null   float64
 2   Gender          10200 non-null  object 
 3   Region          10200 non-null  object 
 4   ContractType    10200 non-null  object 
 5   MonthlyCharges  10200 non-null  float64
 6   SignupDate      10200 non-null  object 
dtypes: float64(2), object(5)
memory usage: 557.9+ KB

USAGE DATA
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   CustomerID   120000 non-null  object 
 1   Month        120000 non-null  object 
 2   CallMinutes  120000 non-null  float64
 3   DataUsageGB  120000 non-null  float64
 4   SMSCount     120000 non-null  int6

In [62]:
# Show all column names clearly

print("customer_info columns:")
print(customer_info.columns.tolist())

print("\nusage_data columns:")
print(usage_data.columns.tolist())

print("\nchurn_labels columns:")
print(churn_labels.columns.tolist())

customer_info columns:
['CustomerID', 'Age', 'Gender', 'Region', 'ContractType', 'MonthlyCharges', 'SignupDate']

usage_data columns:
['CustomerID', 'Month', 'CallMinutes', 'DataUsageGB', 'SMSCount', 'Complaints']

churn_labels columns:
['CustomerID', 'Churn']


In [63]:
# Check unique customer counts in each table

print("Unique customers in customer_info:",
      customer_info["CustomerID"].nunique())

print("Unique customers in usage_data:",
      usage_data["CustomerID"].nunique())

print("Unique customers in churn_labels:",
      churn_labels["CustomerID"].nunique())

Unique customers in customer_info: 10000
Unique customers in usage_data: 10000
Unique customers in churn_labels: 10000


In [64]:
# Check duplicate rows

print("Duplicate rows in customer_info:",
      customer_info.duplicated().sum())

print("Duplicate rows in usage_data:",
      usage_data.duplicated().sum())

print("Duplicate rows in churn_labels:",
      churn_labels.duplicated().sum())

Duplicate rows in customer_info: 195
Duplicate rows in usage_data: 0
Duplicate rows in churn_labels: 0


In [65]:
# Check duplicate customer IDs

print("Duplicate CustomerIDs in customer_info:",
      customer_info["CustomerID"].duplicated().sum())

print("Duplicate CustomerIDs in churn_labels:",
      churn_labels["CustomerID"].duplicated().sum())

Duplicate CustomerIDs in customer_info: 200
Duplicate CustomerIDs in churn_labels: 0


In [66]:
# Convert IDs to Python sets for comparison

info_ids = set(customer_info["CustomerID"])
usage_ids = set(usage_data["CustomerID"])
label_ids = set(churn_labels["CustomerID"])

print("Customers in customer_info but not churn_labels:",
      len(info_ids - label_ids))

print("Customers in churn_labels but not customer_info:",
      len(label_ids - info_ids))

print("Customers in usage_data but not churn_labels:",
      len(usage_ids - label_ids))

print("Customers in churn_labels but not usage_data:",
      len(label_ids - usage_ids))

Customers in customer_info but not churn_labels: 0
Customers in churn_labels but not customer_info: 0
Customers in usage_data but not churn_labels: 0
Customers in churn_labels but not usage_data: 0


In [67]:
# Inspect target distribution

print(churn_labels["Churn"].value_counts())

print("\nChurn percentage:")
print(churn_labels["Churn"].value_counts(normalize=True) * 100)

Churn
0    9214
1     786
Name: count, dtype: int64

Churn percentage:
Churn
0    92.14
1     7.86
Name: proportion, dtype: float64


In [68]:
# Show all rows for CustomerIDs that appear more than once

duplicate_customers = customer_info[
    customer_info["CustomerID"].duplicated(keep=False)      #keep=False returns all duplicates, not just the first or last occurrence
].sort_values("CustomerID")                                 #sort_values() sorts the DataFrame by the "CustomerID" column so that all rows for the same customer are grouped together

display(duplicate_customers)

,CustomerID,Age,Gender,Region,ContractType,MonthlyCharges,SignupDate
1031,CUST_1031,57.0,Male,Urban,Monthly,40.60,2023-05-23
10070,CUST_1031,57.0,Male,Urban,Monthly,40.60,2023-05-23
1042,CUST_1042,NaN,Male,Urban,Monthly,63.09,2021-04-24
10075,CUST_1042,NaN,Male,Urban,Monthly,63.09,2021-04-24
10061,CUST_1128,55.0,Male,Urban,Prepaid,31.88,2019-01-18
...,...,...,...,...,...,...,...
10111,CUST_9881,36.0,Male,Suburban,Yearly,56.38,2018-02-04
10096,CUST_9977,NaN,Male,Urban,Monthly,34.32,2020-03-22
9977,CUST_9977,NaN,Male,Urban,Monthly,34.32,2020-03-22
999,CUST_999,55.0,Female,Urban,Prepaid,37.78,2021-04-16


In [69]:
# Check whether repeated customer records have conflicting values

duplicate_conflicts = (
    duplicate_customers
    .groupby("CustomerID")
    .nunique(dropna=False)      #nunique(false) missing values (NaN) are also counted.
)

display(duplicate_conflicts.head(20))

,Age,Gender,Region,ContractType,MonthlyCharges,SignupDate
CustomerID,,,,,,
CUST_1031,1,1,1,1,1,1
CUST_1042,1,1,1,1,1,1
CUST_1128,1,1,1,1,1,1
CUST_1134,1,1,1,1,1,1
CUST_1228,1,1,1,1,1,1
CUST_1293,1,1,1,1,1,1
CUST_1296,1,1,1,1,1,1
CUST_1306,1,1,1,1,1,1
CUST_1322,1,1,1,1,1,1


In [70]:
#drop duplicate customer records, keeping only the first occurrence

customer_info_clean = customer_info.drop_duplicates()

print("Before:", customer_info.shape)
print("After :", customer_info_clean.shape)

print(
    "Unique customers after cleaning:",
    customer_info_clean["CustomerID"].nunique()
)

Before: (10200, 7)
After : (10005, 7)
Unique customers after cleaning: 10000


In [71]:
# Check for remaining duplicate CustomerIDs after cleaning

remaining_duplicate_ids = (
    customer_info_clean["CustomerID"]
    .duplicated()
    .sum()
)

print("Remaining duplicate CustomerIDs:", remaining_duplicate_ids)

Remaining duplicate CustomerIDs: 5


In [72]:
# Show all rows for CustomerIDs that appear more than once after cleaning

remaining_dupes = customer_info_clean[
    customer_info_clean["CustomerID"].duplicated(keep=False)  #keep=False returns all duplicates, not just the first or last occurrence
].sort_values("CustomerID")

display(remaining_dupes)

,CustomerID,Age,Gender,Region,ContractType,MonthlyCharges,SignupDate
248,CUST_248,22.0,Male,Rural,Monthly,58.85,2022-09-06
10024,CUST_248,22.0,Male,Rural,Monthly,58.85,2022-06-09
308,CUST_308,37.0,Male,Urban,Monthly,42.28,2018-01-03
10144,CUST_308,37.0,Male,Urban,Monthly,42.28,2018-03-01
333,CUST_333,NaN,Male,Rural,Prepaid,63.64,2023-11-02
10089,CUST_333,NaN,Male,Rural,Prepaid,63.64,2023-02-11
391,CUST_391,NaN,Female,Urban,Yearly,31.60,2023-11-03
10150,CUST_391,NaN,Female,Urban,Yearly,31.60,2023-03-11
466,CUST_466,23.0,Female,Urban,Monthly,45.18,2021-03-07
10076,CUST_466,23.0,Female,Urban,Monthly,45.18,2021-07-03


In [73]:
# Check whether repeated customer records have conflicting values after cleaning

remaining_dupe_conflicts = (
    remaining_dupes
    .groupby("CustomerID")
    .nunique(dropna=False)
)

display(remaining_dupe_conflicts)

,Age,Gender,Region,ContractType,MonthlyCharges,SignupDate
CustomerID,,,,,,
CUST_248,1,1,1,1,1,2
CUST_308,1,1,1,1,1,2
CUST_333,1,1,1,1,1,2
CUST_391,1,1,1,1,1,2
CUST_466,1,1,1,1,1,2


In [74]:
# Convert date-like columns to proper datetime values for inspection

customer_info_clean["SignupDate"] = pd.to_datetime(
    customer_info_clean["SignupDate"],
    errors="coerce"                    #errors="coerce" will convert invalid date strings to NaT (Not a Time) instead of raising an error
)

usage_data["Month"] = pd.to_datetime(
    usage_data["Month"],
    errors="coerce"                    #errors="coerce" will convert invalid date strings to NaT (Not a Time) instead of raising an error
)

# Check for missing values in each dataset

first_usage = (
    usage_data
    .groupby("CustomerID")["Month"]
    .min()
    .reset_index()
    .rename(columns={"Month": "FirstUsageMonth"})
)

display(first_usage.head())

# Check for signup date conflicts with first usage month

signup_conflicts = (
    remaining_dupes
    .merge(first_usage, on="CustomerID", how="left")
    .sort_values(["CustomerID", "SignupDate"])
)

display(signup_conflicts)

C:\Users\Windows 10\AppData\Local\Temp\ipykernel_15288\3946334670.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  customer_info_clean["SignupDate"] = pd.to_datetime(


,CustomerID,FirstUsageMonth
0,CUST_0,2023-01-31
1,CUST_1,2023-01-31
2,CUST_10,2023-01-31
3,CUST_100,2023-01-31
4,CUST_1000,2023-01-31


,CustomerID,Age,Gender,Region,ContractType,MonthlyCharges,SignupDate,FirstUsageMonth
1,CUST_248,22.0,Male,Rural,Monthly,58.85,2022-06-09,2023-01-31
0,CUST_248,22.0,Male,Rural,Monthly,58.85,2022-09-06,2023-01-31
2,CUST_308,37.0,Male,Urban,Monthly,42.28,2018-01-03,2023-01-31
3,CUST_308,37.0,Male,Urban,Monthly,42.28,2018-03-01,2023-01-31
5,CUST_333,NaN,Male,Rural,Prepaid,63.64,2023-02-11,2023-01-31
4,CUST_333,NaN,Male,Rural,Prepaid,63.64,2023-11-02,2023-01-31
7,CUST_391,NaN,Female,Urban,Yearly,31.60,2023-03-11,2023-01-31
6,CUST_391,NaN,Female,Urban,Yearly,31.60,2023-11-03,2023-01-31
8,CUST_466,23.0,Female,Urban,Monthly,45.18,2021-03-07,2023-01-31
9,CUST_466,23.0,Female,Urban,Monthly,45.18,2021-07-03,2023-01-31


In [75]:
# First observed usage month for each customer
first_observed_usage = (
    usage_data
    .groupby("CustomerID")["Month"]
    .min()
    .reset_index()
    .rename(columns={"Month": "FirstObservedUsageMonth"})
)

# Join signup date with first observed usage
signup_audit = (
    customer_info_clean
    .merge(first_observed_usage, on="CustomerID", how="left")
)

# Check whether signup occurs after usage is already observed
signup_audit["SignupAfterObservedUsage"] = (
    signup_audit["SignupDate"] >
    signup_audit["FirstObservedUsageMonth"]
)

print(
    signup_audit["SignupAfterObservedUsage"]
    .value_counts()
)

print("\nPercentage:")
print(
    signup_audit["SignupAfterObservedUsage"]
    .value_counts(normalize=True) * 100
)

SignupAfterObservedUsage
False    9252
True      753
Name: count, dtype: int64

Percentage:
SignupAfterObservedUsage
False    92.473763
True      7.526237
Name: proportion, dtype: float64


In [76]:
suspicious_signup = signup_audit[
    signup_audit["SignupAfterObservedUsage"]
]

display(
    suspicious_signup[
        [
            "CustomerID",
            "SignupDate",
            "FirstObservedUsageMonth"
        ]
    ].head(20)
)

,CustomerID,SignupDate,FirstObservedUsageMonth
14,CUST_14,2023-06-04,2023-01-31
32,CUST_32,2023-03-02,2023-01-31
64,CUST_64,2023-06-16,2023-01-31
76,CUST_76,2023-04-26,2023-01-31
79,CUST_79,2023-05-25,2023-01-31
89,CUST_89,2023-02-06,2023-01-31
97,CUST_97,2023-05-05,2023-01-31
110,CUST_110,2023-04-17,2023-01-31
114,CUST_114,2023-03-24,2023-01-31
116,CUST_116,2023-06-24,2023-01-31


In [77]:
# Sort each customer by signup date,
# then retain the earliest record for duplicated CustomerIDs

customer_info_clean = (
    customer_info_clean
    .sort_values(["CustomerID", "SignupDate"])
    .drop_duplicates(
        subset="CustomerID",
        keep="first"
    )
    .reset_index(drop=True)
)

print("Final shape:", customer_info_clean.shape)
print("Unique customers:", customer_info_clean["CustomerID"].nunique())
print(
    "Remaining duplicate IDs:",
    customer_info_clean["CustomerID"].duplicated().sum()
)

Final shape: (10000, 7)
Unique customers: 10000
Remaining duplicate IDs: 0


In [78]:
# Rebuild signup-date audit after deduplication

signup_audit = (
    customer_info_clean
    .merge(
        first_observed_usage,
        on="CustomerID",
        how="left"
    )
)

signup_audit["SignupAfterObservedUsage"] = (
    signup_audit["SignupDate"]
    > signup_audit["FirstObservedUsageMonth"]
)

print(
    signup_audit["SignupAfterObservedUsage"]
    .value_counts()
)

print("\nPercentage:")
print(
    signup_audit["SignupAfterObservedUsage"]
    .value_counts(normalize=True) * 100
)




SignupAfterObservedUsage
False    9249
True      751
Name: count, dtype: int64

Percentage:
SignupAfterObservedUsage
False    92.49
True      7.51
Name: proportion, dtype: float64


In [79]:
# Number of usage records per customer

usage_records_per_customer = (
    usage_data
    .groupby("CustomerID")
    .size()
)

print(usage_records_per_customer.describe())

count    10000.0
mean        12.0
std          0.0
min         12.0
25%         12.0
50%         12.0
75%         12.0
max         12.0
dtype: float64


In [80]:
# Distribution of the number of monthly records per customer

print(
    usage_records_per_customer
    .value_counts()
    .sort_index()
)

print("Earliest usage month:", usage_data["Month"].min())
print("Latest usage month:", usage_data["Month"].max())

print("\nUnique months:")
print(
    sorted(usage_data["Month"].dropna().unique())
)

12    10000
Name: count, dtype: int64
Earliest usage month: 2023-01-31 00:00:00
Latest usage month: 2023-12-31 00:00:00

Unique months:
[Timestamp('2023-01-31 00:00:00'), Timestamp('2023-02-28 00:00:00'), Timestamp('2023-03-31 00:00:00'), Timestamp('2023-04-30 00:00:00'), Timestamp('2023-05-31 00:00:00'), Timestamp('2023-06-30 00:00:00'), Timestamp('2023-07-31 00:00:00'), Timestamp('2023-08-31 00:00:00'), Timestamp('2023-09-30 00:00:00'), Timestamp('2023-10-31 00:00:00'), Timestamp('2023-11-30 00:00:00'), Timestamp('2023-12-31 00:00:00')]


In [81]:
# Check for missing values in the Age column of customer_info_clean

age_missing_count = customer_info_clean["Age"].isna().sum()
age_missing_pct = customer_info_clean["Age"].isna().mean() * 100

print("Missing Age count:", age_missing_count)
print(f"Missing Age percentage: {age_missing_pct:.2f}%")

# Create a new column to indicate whether Age is missing (1) or not (0)

customer_info_clean["AgeMissing"] = (
    customer_info_clean["Age"]
    .isna()
    .astype(int)
)

customer_info_clean["AgeMissing"].value_counts()

# Analyze the relationship between missing Age values and churn

age_missing_churn = (
    customer_info_clean[
        ["CustomerID", "AgeMissing"]
    ]
    .merge(
        churn_labels,
        on="CustomerID",
        how="left"
    )
)

age_missing_churn.groupby("AgeMissing")["Churn"].agg(
    customer_count="count",
    churn_count="sum",
    churn_rate="mean"
)



Missing Age count: 3500
Missing Age percentage: 35.00%


,customer_count,churn_count,churn_rate
AgeMissing,,,
0,6500,506,0.077846
1,3500,280,0.080000


In [82]:
# Analyze the relationship between missing Age values and other categorical variables

for column in ["Gender", "Region", "ContractType"]:
    print(f"\n--- {column} ---")

    result = pd.crosstab(
        customer_info_clean[column],
        customer_info_clean["AgeMissing"],
        normalize="index"
    ) * 100

    display(result.round(2))


--- Gender ---


AgeMissing,0,1
Gender,,
Female,64.10,35.90
Male,65.91,34.09



--- Region ---


AgeMissing,0,1
Region,,
Rural,65.30,34.70
Suburban,64.70,35.30
Urban,65.01,34.99



--- ContractType ---


AgeMissing,0,1
ContractType,,
Monthly,65.42,34.58
Prepaid,64.59,35.41
Yearly,65.00,35.00


In [83]:
customer_info_clean["Age"].describe()

count    6500.000000
mean       43.419385
std        14.967374
min        18.000000
25%        31.000000
50%        43.000000
75%        56.000000
max        69.000000
Name: Age, dtype: float64

In [84]:
# Show the 10 youngest and 10 oldest customers

print(
    customer_info_clean["Age"]
    .sort_values()
    .head(10)
)

print(
    customer_info_clean["Age"]
    .sort_values(ascending=False)
    .head(10)
)

6363    18.0
6365    18.0
9558    18.0
162     18.0
7742    18.0
9039    18.0
9040    18.0
5259    18.0
9042    18.0
8353    18.0
Name: Age, dtype: float64
623     69.0
2059    69.0
669     69.0
7240    69.0
3261    69.0
7251    69.0
9020    69.0
5432    69.0
5438    69.0
7278    69.0
Name: Age, dtype: float64


In [85]:
# Inspect unique values and frequencies in categorical columns

categorical_columns = [
    "Gender",
    "Region",
    "ContractType"
]

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(
        customer_info_clean[column]
        .value_counts(dropna=False)          #dropna=False ensures that missing values (NaN) are also counted in the frequency distribution
    )


--- Gender ---
Gender
Female    5022
Male      4978
Name: count, dtype: int64

--- Region ---
Region
Suburban    3402
Rural       3311
Urban       3287
Name: count, dtype: int64

--- ContractType ---
ContractType
Prepaid    4041
Monthly    3953
Yearly     2006
Name: count, dtype: int64


In [86]:
# Statistical summary of monthly charges

customer_info_clean["MonthlyCharges"].describe()

count    10000.000000
mean        51.000240
std         21.543385
min         -6.720000
25%         39.797500
50%         50.015000
75%         60.500000
max        390.300000
Name: MonthlyCharges, dtype: float64

In [87]:
# Look for impossible or suspicious charge values

print(
    "Zero or negative MonthlyCharges:",
    (customer_info_clean["MonthlyCharges"] <= 0).sum()
)

print(
    "Minimum MonthlyCharges:",
    customer_info_clean["MonthlyCharges"].min()
)

print(
    "Maximum MonthlyCharges:",
    customer_info_clean["MonthlyCharges"].max()
)

Zero or negative MonthlyCharges: 5
Minimum MonthlyCharges: -6.72
Maximum MonthlyCharges: 390.3


In [88]:
## Statistical summary of usage data

usage_columns = [
    "CallMinutes",
    "DataUsageGB",
    "SMSCount",
    "Complaints"
]

usage_data[usage_columns].describe()

,CallMinutes,DataUsageGB,SMSCount,Complaints
count,120000.000000,120000.000000,120000.000000,120000.000000
mean,492.401958,9.898117,24.525392,0.503000
std,117.443986,3.067815,14.423944,0.708763
min,34.140000,0.000000,0.000000,0.000000
25%,418.450000,7.810000,12.000000,0.000000
50%,494.960000,9.920000,25.000000,0.000000
75%,571.080000,11.980000,37.000000,1.000000
max,914.160000,22.730000,49.000000,6.000000


In [89]:
# Check for impossible negative values

for column in usage_columns:
    negative_count = (usage_data[column] < 0).sum()

    print(
        f"{column}: {negative_count} negative values"
    )

CallMinutes: 0 negative values
DataUsageGB: 0 negative values
SMSCount: 0 negative values
Complaints: 0 negative values


In [90]:
for column in usage_columns:
    print(
        f"{column}: "
        f"min={usage_data[column].min()}, "
        f"max={usage_data[column].max()}"
    )

CallMinutes: min=34.14, max=914.16
DataUsageGB: min=0.0, max=22.73
SMSCount: min=0, max=49
Complaints: min=0, max=6


In [91]:
# Look for impossible or suspicious charge values

invalid_charges = customer_info_clean[
    customer_info_clean["MonthlyCharges"] <= 0
]

display(invalid_charges)

,CustomerID,Age,Gender,Region,ContractType,MonthlyCharges,SignupDate,AgeMissing
4081,CUST_4671,56.0,Male,Rural,Prepaid,-3.89,2020-11-12,0
5085,CUST_5575,39.0,Female,Suburban,Monthly,-0.46,2023-02-01,0
6469,CUST_6820,60.0,Male,Urban,Yearly,-3.15,2018-02-09,0
8409,CUST_8567,36.0,Male,Rural,Prepaid,-6.72,2018-04-17,0
9339,CUST_9403,19.0,Male,Rural,Monthly,-4.55,2023-05-18,0


In [92]:
## Join invalid charges with churn labels to see if there's any correlation

invalid_charges_with_target = (
    invalid_charges
    .merge(
        churn_labels,
        on="CustomerID",
        how="left"
    )
)

display(invalid_charges_with_target)

,CustomerID,Age,Gender,Region,ContractType,MonthlyCharges,SignupDate,AgeMissing,Churn
0,CUST_4671,56.0,Male,Rural,Prepaid,-3.89,2020-11-12,0,0
1,CUST_5575,39.0,Female,Suburban,Monthly,-0.46,2023-02-01,0,0
2,CUST_6820,60.0,Male,Urban,Yearly,-3.15,2018-02-09,0,0
3,CUST_8567,36.0,Male,Rural,Prepaid,-6.72,2018-04-17,0,0
4,CUST_9403,19.0,Male,Rural,Monthly,-4.55,2023-05-18,0,0


In [93]:
## Check for outliers in MonthlyCharges using the IQR method
Q1 = customer_info_clean["MonthlyCharges"].quantile(0.25)
Q3 = customer_info_clean["MonthlyCharges"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

Q1: 39.7975
Q3: 60.5
IQR: 20.7025
Lower bound: 8.743749999999999
Upper bound: 91.55375000000001


In [94]:
# Check for outliers in MonthlyCharges using the IQR method

monthly_charge_outliers = customer_info_clean[(customer_info_clean["MonthlyCharges"] < lower_bound)|(customer_info_clean["MonthlyCharges"] > upper_bound)]

print("MonthlyCharges outlier count:", len(monthly_charge_outliers))

display(
    monthly_charge_outliers
    .sort_values("MonthlyCharges")
    .head(10)
)

display(
    monthly_charge_outliers
    .sort_values("MonthlyCharges", ascending=False)
    .head(10)
)

MonthlyCharges outlier count: 111


,CustomerID,Age,Gender,Region,ContractType,MonthlyCharges,SignupDate,AgeMissing
8409,CUST_8567,36.0,Male,Rural,Prepaid,-6.72,2018-04-17,0
9339,CUST_9403,19.0,Male,Rural,Monthly,-4.55,2023-05-18,0
4081,CUST_4671,56.0,Male,Rural,Prepaid,-3.89,2020-11-12,0
6469,CUST_6820,60.0,Male,Urban,Yearly,-3.15,2018-02-09,0
5085,CUST_5575,39.0,Female,Suburban,Monthly,-0.46,2023-02-01,0
4984,CUST_5484,NaN,Female,Suburban,Monthly,0.38,2022-09-18,1
935,CUST_184,61.0,Male,Suburban,Monthly,0.72,2021-06-23,0
4833,CUST_5348,47.0,Male,Urban,Prepaid,1.12,2018-11-15,0
836,CUST_1750,51.0,Male,Rural,Monthly,2.83,2023-06-12,0
1311,CUST_2178,59.0,Male,Suburban,Prepaid,3.24,2022-08-23,0


,CustomerID,Age,Gender,Region,ContractType,MonthlyCharges,SignupDate,AgeMissing
5368,CUST_583,NaN,Female,Suburban,Prepaid,390.30,2018-01-07,1
9340,CUST_9404,NaN,Male,Urban,Monthly,369.55,2019-11-19,1
2964,CUST_3666,21.0,Male,Urban,Prepaid,367.65,2021-09-09,0
5304,CUST_5772,35.0,Male,Urban,Monthly,356.00,2023-06-09,0
9867,CUST_988,31.0,Male,Rural,Prepaid,353.35,2022-12-25,0
8460,CUST_8612,22.0,Male,Urban,Prepaid,338.10,2018-01-24,0
8574,CUST_8715,NaN,Male,Suburban,Prepaid,329.95,2023-05-30,1
6163,CUST_6545,62.0,Male,Urban,Yearly,321.45,2023-01-10,0
1286,CUST_2155,48.0,Female,Rural,Monthly,318.50,2021-05-11,0
4696,CUST_5224,NaN,Female,Rural,Yearly,316.80,2021-02-09,1


In [95]:
# Function to summarize outliers in a column using the IQR method
def iqr_outlier_summary(df, column):
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)

    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_count = ((df[column] < lower)|(df[column] > upper)).sum()

    return {
        "column": column,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": outlier_count,
        "outlier_pct": outlier_count / len(df) * 100
    }

In [96]:
# Summarize outliers for usage data columns
usage_outlier_results = []

for column in ["CallMinutes","DataUsageGB", "SMSCount","Complaints"]:
    usage_outlier_results.append(iqr_outlier_summary(usage_data,column))

usage_outlier_summary = pd.DataFrame(usage_outlier_results)

display(usage_outlier_summary)

,column,lower_bound,upper_bound,outlier_count,outlier_pct
0,CallMinutes,189.505,800.025,1611,1.3425
1,DataUsageGB,1.555,18.235,681,0.5675
2,SMSCount,-25.500,74.500,0,0.0000
3,Complaints,-1.500,2.500,1746,1.4550


In [97]:
# Create an audit-only classification of MonthlyCharges values

customer_info_clean["ChargeQualityGroup"] = np.select([customer_info_clean["MonthlyCharges"] <= 0,
                                                      ((customer_info_clean["MonthlyCharges"] > 0)& (customer_info_clean["MonthlyCharges"] < lower_bound)),
                                                      customer_info_clean["MonthlyCharges"] > upper_bound],["Invalid","Low Outlier","High Outlier"],default="Normal")

customer_info_clean["ChargeQualityGroup"].value_counts()

ChargeQualityGroup
Normal          9889
High Outlier      82
Low Outlier       24
Invalid            5
Name: count, dtype: int64

In [ ]:
## Join charge quality groups with churn labels to analyze potential correlations

charge_audit = (
    customer_info_clean[
        [
            "CustomerID",
            "MonthlyCharges",
            "ContractType",
            "ChargeQualityGroup"
        ]
    ]
    .merge(
        churn_labels,
        on="CustomerID",
        how="left"
    )
)

charge_group_summary = (
    charge_audit
    .groupby("ChargeQualityGroup")
    .agg(
        customers=("CustomerID", "count"),
        avg_charge=("MonthlyCharges", "mean"),
        churners=("Churn", "sum"),
        churn_rate=("Churn", "mean")
    )
    .sort_values("customers", ascending=False)
)

display(charge_group_summary)

,customers,avg_charge,churners,churn_rate
ChargeQualityGroup,,,,
Normal,9889,49.953505,779,0.078774
High Outlier,82,193.902927,6,0.073171
Low Outlier,24,5.455000,1,0.041667
Invalid,5,-3.754000,0,0.000000


In [101]:
customer_info_clean["MonthlyCharges"].quantile(
    [0.001, 0.005, 0.01, 0.05, 0.50, 0.95, 0.99, 0.995, 0.999]
)

0.001      3.61962
0.005     10.51960
0.010     14.26980
0.050     24.96800
0.500     50.01500
0.950     75.71050
0.990     89.42070
0.995    107.14970
0.999    310.75605
Name: MonthlyCharges, dtype: float64

In [ ]:
# Create a business-rule flag for clearly invalid charges

customer_info_clean["MonthlyChargesInvalid"] = (
    customer_info_clean["MonthlyCharges"] <= 0
).astype(int)

# Replace impossible negative/zero charges with missing values.
customer_info_clean.loc[
    customer_info_clean["MonthlyCharges"] <= 0,
    "MonthlyCharges"
] = np.nan

print(
    customer_info_clean["MonthlyChargesInvalid"]
    .value_counts()
)

print(
    "Missing MonthlyCharges after invalid-value replacement:",
    customer_info_clean["MonthlyCharges"].isna().sum()
)

MonthlyChargesInvalid
0    9995
1       5
Name: count, dtype: int64
Missing MonthlyCharges after invalid-value replacement: 5


In [103]:
# A customer should have only one record for each month

duplicate_customer_month = usage_data.duplicated(
    subset=["CustomerID", "Month"]
).sum()

print(
    "Duplicate CustomerID-Month combinations:",
    duplicate_customer_month
)

Duplicate CustomerID-Month combinations: 0


In [105]:
# Count unique observed months for every customer

months_per_customer = (
    usage_data
    .groupby("CustomerID")["Month"]
    .nunique()
)

print(months_per_customer.value_counts())

Month
12    10000
Name: count, dtype: int64


In [106]:
usage_with_churn = (
    usage_data
    .merge(
        churn_labels,
        on="CustomerID",
        how="left"
    )
)

print(usage_with_churn.shape)
display(usage_with_churn.head())

(120000, 7)


,CustomerID,Month,CallMinutes,DataUsageGB,SMSCount,Complaints,Churn
0,CUST_0,2023-01-31,518.12,7.21,9,1,0
1,CUST_0,2023-02-28,439.79,5.99,35,1,0
2,CUST_0,2023-03-31,545.06,14.16,14,1,0
3,CUST_0,2023-04-30,461.78,6.66,40,0,0
4,CUST_0,2023-05-31,434.82,12.75,26,2,0


In [108]:
# Analyze average usage behavior by month and churn status
monthly_behavior = (
    usage_with_churn
    .groupby(["Month", "Churn"])
    .agg(
        avg_call_minutes=("CallMinutes", "mean"),
        avg_data_usage=("DataUsageGB", "mean"),
        avg_sms=("SMSCount", "mean"),
        avg_complaints=("Complaints", "mean")
    )
    .reset_index()
)

display(monthly_behavior)

,Month,Churn,avg_call_minutes,avg_data_usage,avg_sms,avg_complaints
0,2023-01-31,0,498.807339,9.996943,24.534947,0.513024
1,2023-01-31,1,503.195115,9.871565,25.234097,0.478372
2,2023-02-28,0,497.848476,9.994553,24.296722,0.500000
3,2023-02-28,1,499.606132,10.167443,25.122137,0.536896
4,2023-03-31,0,497.674652,10.018955,24.579119,0.506295
5,2023-03-31,1,498.578435,9.977137,24.679389,0.547074
6,2023-04-30,0,498.580097,9.947096,24.566529,0.502388
7,2023-04-30,1,500.094567,9.945013,24.298982,0.522901
8,2023-05-31,0,499.022163,10.022918,24.572498,0.501845
9,2023-05-31,1,501.146883,9.966247,25.010178,0.517812


In [109]:
# Create numeric month index: Jan=1, ..., Dec=12

usage_with_churn["MonthNumber"] = (
    usage_with_churn["Month"].dt.month
)

display(
    usage_with_churn[
        ["Month", "MonthNumber"]
    ].drop_duplicates().sort_values("MonthNumber")
)

,Month,MonthNumber
0,2023-01-31,1
1,2023-02-28,2
2,2023-03-31,3
3,2023-04-30,4
4,2023-05-31,5
5,2023-06-30,6
6,2023-07-31,7
7,2023-08-31,8
8,2023-09-30,9
9,2023-10-31,10


In [ ]:
# Analyze early vs. recent usage behavior for each customer
early_usage = (
    usage_with_churn[
        usage_with_churn["MonthNumber"] <= 9
    ]
    .groupby("CustomerID")
    .agg(
        early_call_avg=("CallMinutes", "mean"),
        early_data_avg=("DataUsageGB", "mean"),
        early_sms_avg=("SMSCount", "mean"),
        early_complaints_avg=("Complaints", "mean")
    )
    .reset_index()
)


recent_usage = (
    usage_with_churn[
        usage_with_churn["MonthNumber"] >= 10
    ]
    .groupby("CustomerID")
    .agg(
        recent_call_avg=("CallMinutes", "mean"),
        recent_data_avg=("DataUsageGB", "mean"),
        recent_sms_avg=("SMSCount", "mean"),
        recent_complaints_avg=("Complaints", "mean")
    )
    .reset_index()
)

behavior_change = (
    early_usage
    .merge(
        recent_usage,
        on="CustomerID",
        how="inner"
    )
    .merge(
        churn_labels,
        on="CustomerID",
        how="left"
    )
)

print(behavior_change.shape)
display(behavior_change.head())


(10000, 10)


,CustomerID,early_call_avg,early_data_avg,early_sms_avg,early_complaints_avg,recent_call_avg,recent_data_avg,recent_sms_avg,recent_complaints_avg,Churn
0,CUST_0,493.412222,9.285556,26.000000,0.666667,488.696667,11.326667,22.000000,0.333333,0
1,CUST_1,457.194444,10.083333,32.000000,0.888889,436.446667,10.943333,42.333333,1.666667,0
2,CUST_10,617.418889,8.223333,18.333333,1.222222,661.150000,9.740000,37.333333,0.333333,0
3,CUST_100,529.485556,9.123333,30.444444,0.555556,549.700000,10.980000,24.666667,0.000000,0
4,CUST_1000,506.097778,10.765556,21.111111,1.111111,457.233333,11.360000,41.666667,0.666667,0


In [ ]:
# Calculate the change in usage behavior between early and recent months for each customer

behavior_change["call_change"] = (
    behavior_change["recent_call_avg"]
    - behavior_change["early_call_avg"]
)

behavior_change["data_change"] = (
    behavior_change["recent_data_avg"]
    - behavior_change["early_data_avg"]
)

behavior_change["sms_change"] = (
    behavior_change["recent_sms_avg"]
    - behavior_change["early_sms_avg"]
)

behavior_change["complaints_change"] = (
    behavior_change["recent_complaints_avg"]
    - behavior_change["early_complaints_avg"]
)

# Summarize the average change in usage behavior by churn status

change_summary = (
    behavior_change
    .groupby("Churn")
    .agg(
        customers=("CustomerID", "count"),

        avg_call_change=("call_change", "mean"),
        avg_data_change=("data_change", "mean"),
        avg_sms_change=("sms_change", "mean"),
        avg_complaints_change=("complaints_change", "mean")
    )
)

display(change_summary)

,customers,avg_call_change,avg_data_change,avg_sms_change,avg_complaints_change
Churn,,,,,
0,9214,-25.157070,-0.420433,-0.002110,0.000422
1,786,-23.218569,-0.381039,-0.229008,-0.023042


In [ ]:
# Calculate percentage change in usage behavior for each customer
behavior_change["call_change_pct"] = (
    behavior_change["call_change"]
    / behavior_change["early_call_avg"]
) * 100

behavior_change["data_change_pct"] = (
    behavior_change["data_change"]
    / behavior_change["early_data_avg"]
) * 100


pct_change_summary = (
    behavior_change
    .groupby("Churn")
    .agg(
        avg_call_change_pct=("call_change_pct", "mean"),
        avg_data_change_pct=("data_change_pct", "mean")
    )
)

display(pct_change_summary)

,avg_call_change_pct,avg_data_change_pct
Churn,,
0,-4.921113,-3.208118
1,-4.449991,-2.783477
